# 🚀 TASK 6.3-C: CHIẾN DỊCH HUẤN LUYỆN THÍCH ỨNG ĐA MIỀN TRÊN MÁY CHỦ KAGGLE
### Huấn luyện 3 nhánh ứng viên M3 (+Data), M4 (+Distillation), M5 (+LoRA) — Milestone 6 (Offline 100%)

> **Mã nhiệm vụ**: `Task 6.3-C` *(Milestone 6 - Tuần 6)*  
> **Người chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Người phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường thực thi**: Kaggle Server GPU (NVIDIA T4 / P100 / RTX PRO 6000 Blackwell, Internet: Disabled)  
> **Kế thừa hạ tầng**: Kho dữ liệu Task 6.2; Hàm mất mát Distillation & Consistency Task 6.3-B; Khóa cố định 18 tập test Task 5.1  

---
### 📌 HƯỚNG DẪN CÁC DATASET ĐẦU VÀO CẦN THIẾT (KAGGLE INPUT):
1. **`fatformer-code`**: Dataset chứa mã nguồn dự án (tải tệp `fatformer_code.zip` phiên bản vừa cập nhật lên).
2. **`fatformer-assets-pack`**: Dataset chứa trọng số `ViT-L-14.pt`, `fatformer_4class_ckpt.pth`, checkpoint Plan A và từ vựng BPE.

*(Lưu ý: 18 tập test benchmark của Task 5.1 được khóa cố định, KHÔNG nạp vào task này nhằm bảo vệ chống rò rỉ dữ liệu).*

In [ ]:
# ==============================================================================
# 1. THIẾT LẬP MÃ NGUỒN VÀ GIẢI NÉN OFFLINE TRÊN KAGGLE
# ==============================================================================
import os
import sys
import glob
import shutil
import zipfile
import subprocess

print("=" * 80)
print("BƯỚC 1: THIẾT LẬP MÃ NGUỒN FATFORMER-XLA TRÊN KAGGLE")
print("=" * 80)

WORK_DIR = "/kaggle/working/fatformer-xla"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# 1. Định vị và nạp mã nguồn từ Kaggle Input (hỗ trợ cả tệp zip và thư mục giải nén sẵn)
zip_candidates = glob.glob("/kaggle/input/**/fatformer_code.zip", recursive=True) + \
                 glob.glob("/kaggle/input/**/fatformer-code*.zip", recursive=True)
train_py_files = glob.glob("/kaggle/input/**/train.py", recursive=True)
code_dirs = glob.glob("/kaggle/input/**/train_crossdomain.py", recursive=True)

if zip_candidates:
    src_zip = zip_candidates[0]
    print(f"[*] Giải nén mã nguồn từ gói: {src_zip}")
    with zipfile.ZipFile(src_zip, 'r') as zf:
        zf.extractall(WORK_DIR)
elif train_py_files:
    root_cand = os.path.dirname(train_py_files[0])
    print(f"[*] Sao chép mã nguồn từ thư mục giải nén sẵn: {root_cand}")
    shutil.copytree(root_cand, WORK_DIR, dirs_exist_ok=True)
elif code_dirs:
    root_cand = os.path.abspath(os.path.join(os.path.dirname(code_dirs[0]), "..", ".."))
    print(f"[*] Sao chép mã nguồn từ thư mục input: {root_cand}")
    shutil.copytree(root_cand, WORK_DIR, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Chưa tìm thấy mã nguồn! Vui lòng Add Input dataset 'fatformer-code'.")

# 2. Xử lý từ vựng BPE Tokenizer offline (đảm bảo CLIP nạp hoàn toàn không cần Internet)
clip_dir = os.path.join(WORK_DIR, "src", "models", "clip")
os.makedirs(clip_dir, exist_ok=True)
target_bpe_gz = os.path.join(clip_dir, "bpe_simple_vocab_16e6.txt.gz")

bpe_candidates_gz = glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt.gz", recursive=True)
bpe_candidates_txt = glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt", recursive=True)

if not os.path.exists(target_bpe_gz):
    if bpe_candidates_gz:
        shutil.copy2(bpe_candidates_gz[0], target_bpe_gz)
        print(f"[✓] Đã nạp BPE vocab .gz từ: {bpe_candidates_gz[0]}")
    elif bpe_candidates_txt:
        subprocess.run(f"gzip -c '{bpe_candidates_txt[0]}' > '{target_bpe_gz}'", shell=True, check=True)
        print(f"[✓] Đã nén và nạp BPE vocab từ: {bpe_candidates_txt[0]}")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")


In [ ]:
# ==============================================================================
# 2. LIÊN KẾT TÀI NGUYÊN PRETRAINED & CHECKPOINTS MỎ NEO
# ==============================================================================
import os
import glob
import shutil

print("=" * 80)
print("BƯỚC 2: TỰ ĐỘNG NHẬN DIỆN VÀ LIÊN KẾT CHECKPOINTS MỎ NEO")
print("=" * 80)

os.makedirs(os.path.join(WORK_DIR, "DATASET", "pretrained"), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "DATASET", "checkpoints"), exist_ok=True)

# 1. Tìm kiếm Backbone CLIP ViT-L-14.pt
clip_files = glob.glob("/kaggle/input/**/ViT-L-14.pt", recursive=True)
if clip_files:
    src_clip = clip_files[0]
    dst_clip = os.path.join(WORK_DIR, "DATASET", "pretrained", "ViT-L-14.pt")
    if not os.path.exists(dst_clip):
        try:
            os.symlink(src_clip, dst_clip)
        except Exception:
            shutil.copy2(src_clip, dst_clip)
    print(f"[✓] Đã liên kết CLIP Backbone: {src_clip} -> {dst_clip}")
else:
    print("[!] Cảnh báo: Chưa tìm thấy ViT-L-14.pt trong input. Hãy Add Input dataset fatformer-assets-pack.")

# 2. Tìm kiếm Checkpoint khởi tạo Student (Plan A) với mẫu quét mở rộng toàn diện
plan_a_candidates = (
    glob.glob("/kaggle/input/fatformer_plan_a_best_weights/**/*.pth", recursive=True) +
    glob.glob("/kaggle/input/fatformer-plan-a-best-weights/**/*.pth", recursive=True) +
    glob.glob("/kaggle/input/**/*fatformer*plan_a*best*weights*.pth", recursive=True) +
    glob.glob("/kaggle/input/**/*fatformer*plan_a*final*.pth", recursive=True) +
    glob.glob("/kaggle/input/**/*fatformer*plan_a*best*.pth", recursive=True) +
    glob.glob("/kaggle/input/**/*model_best*.pth", recursive=True) +
    glob.glob("/kaggle/input/**/*fatformer_plan_a*.pth", recursive=True) +
    glob.glob("/kaggle/input/**/checkpoint*/*.pth", recursive=True)
)

# Lọc chỉ chọn các tệp checkpoint thực tế dung lượng lớn (> 500MB)
valid_plan_a = [c for c in plan_a_candidates if os.path.isfile(c) and os.path.getsize(c) > 500 * (1024**2)]

if valid_plan_a:
    src_init = valid_plan_a[0]
    dst_init = os.path.join(WORK_DIR, "DATASET", "checkpoints", "fatformer_plan_a_best_weights.pth")
    if not os.path.exists(dst_init):
        try:
            os.symlink(src_init, dst_init)
        except Exception:
            shutil.copy2(src_init, dst_init)
    file_size_mb = os.path.getsize(src_init) / (1024**2)
    print(f"[✓] Đã liên kết Checkpoint Student Plan A: {src_init} ({file_size_mb:.2f} MB) -> {dst_init}")
else:
    print("[!] Cảnh báo: Chưa tìm thấy checkpoint Plan A trong /kaggle/input!")
    print("    Vui lòng Add Input dataset checkpoint Plan A hoặc Notebook output của Task 4.3.")

# 3. Tìm kiếm Checkpoint Teacher mỏ neo (fatformer_4class_ckpt.pth)
teacher_files = glob.glob("/kaggle/input/**/fatformer_4class_ckpt.pth", recursive=True)
if teacher_files:
    src_teacher = teacher_files[0]
    dst_teacher = os.path.join(WORK_DIR, "DATASET", "pretrained", "fatformer_4class_ckpt.pth")
    if not os.path.exists(dst_teacher):
        try:
            os.symlink(src_teacher, dst_teacher)
        except Exception:
            shutil.copy2(src_teacher, dst_teacher)
    print(f"[✓] Đã liên kết Checkpoint Teacher: {src_teacher} -> {dst_teacher}")
else:
    print("[!] Cảnh báo: Chưa tìm thấy fatformer_4class_ckpt.pth, nhánh M4/M5 sẽ dùng Student đóng băng làm Teacher.")


In [ ]:
# ==============================================================================
# 3. KIỂM TRA THIẾT BỊ GPU & SMOKE TEST AUTOGRAD (DRY-RUN)
# ==============================================================================
import torch
import subprocess

print("=" * 80)
print("BƯỚC 3: KIỂM TOÁN PHẦN CỨNG GPU & SMOKE GATE MILESTONE 6")
print("=" * 80)

print(f"[*] Phiên bản PyTorch: {torch.__version__}")
print(f"[*] CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[*] GPU đang kích hoạt: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[!] CẢNH BÁO: Không tìm thấy GPU! Hãy chuyển Kaggle Accelerator sang GPU T4 x2 hoặc P100.")

# Chạy kiểm thử Smoke Test autograd 1 step nhanh để đảm bảo 100% khớp nối
smoke_cmd = "python -u src/training/train_crossdomain.py --dry_run --device cpu --epochs 1 --batch_size 4"
print(f"[*] Khởi chạy kiểm thử smoke gate: {smoke_cmd}")
res = subprocess.run(smoke_cmd, shell=True, capture_output=False)
if res.returncode == 0:
    print("[✓] SMOKE TEST PASS 100%! HỆ THỐNG SẴN SÀNG CHO CHIẾN DỊCH HUẤN LUYỆN CHÍNH THỨC.")
else:
    raise RuntimeError("[X] Smoke test thất bại! Vui lòng kiểm tra log lỗi bên trên.")


In [ ]:
# ==============================================================================
# 4. HUẤN LUYEN NHÁNH M3 — DATA ONLY (5 EPOCHS)
# ==============================================================================
import time
import subprocess

print("=" * 80)
print("BƯỚC 4: HUẤN LUYỆN ỨNG VIÊN NHÁNH M3 — DATA ONLY (5 EPOCHS)")
print("=" * 80)

train_script = "src/training/train_crossdomain.py"
cfg_m3 = "configs/train_m3_data.yaml"
device_flag = "--device cuda" if torch.cuda.is_available() else "--device cpu --dry_run"

cmd_m3 = f"python -u {train_script} --config {cfg_m3} {device_flag}"
print(f"[*] Thực thi: {cmd_m3}")
t0 = time.time()
ret_m3 = subprocess.run(cmd_m3, shell=True).returncode
elapsed_m3 = time.time() - t0

if ret_m3 != 0:
    raise RuntimeError(f"[X] Huấn luyện nhánh M3 thất bại với exit code: {ret_m3}!")
else:
    print(f"[✓] Huấn luyện nhánh M3 hoàn tất thành công trong {elapsed_m3:.2f}s ({elapsed_m3/60:.2f} phút)!")


In [ ]:
# ==============================================================================
# 5. HUẤN LUYỆN NHÁNH M4 — DATA + DISTILLATION + CONSISTENCY (5 EPOCHS)
# ==============================================================================
import time
import subprocess

print("=" * 80)
print("BƯỚC 5: HUẤN LUYỆN ỨNG VIÊN NHÁNH M4 — DATA + DISTILL + CONSISTENCY (5 EPOCHS)")
print("=" * 80)

cfg_m4 = "configs/train_m4_distill.yaml"
device_flag = "--device cuda" if torch.cuda.is_available() else "--device cpu --dry_run"

cmd_m4 = f"python -u {train_script} --config {cfg_m4} {device_flag}"
print(f"[*] Thực thi: {cmd_m4}")
t0 = time.time()
ret_m4 = subprocess.run(cmd_m4, shell=True).returncode
elapsed_m4 = time.time() - t0

if ret_m4 != 0:
    raise RuntimeError(f"[X] Huấn luyện nhánh M4 thất bại với exit code: {ret_m4}!")
else:
    print(f"[✓] Huấn luyện nhánh M4 hoàn tất thành công trong {elapsed_m4:.2f}s ({elapsed_m4/60:.2f} phút)!")


In [ ]:
# ==============================================================================
# 6. HUẤN LUYỆN NHÁNH M5 — M4 + LORA VIT ADAPTER (5 EPOCHS)
# ==============================================================================
import time
import subprocess

print("=" * 80)
print("BƯỚC 6: HUẤN LUYỆN ỨNG VIÊN NHÁNH M5 — M4 + LORA ADAPTER (5 EPOCHS)")
print("=" * 80)

cfg_m5 = "configs/train_m5_lora.yaml"
device_flag = "--device cuda" if torch.cuda.is_available() else "--device cpu --dry_run"

cmd_m5 = f"python -u {train_script} --config {cfg_m5} {device_flag}"
print(f"[*] Thực thi: {cmd_m5}")
t0 = time.time()
ret_m5 = subprocess.run(cmd_m5, shell=True).returncode
elapsed_m5 = time.time() - t0

if ret_m5 != 0:
    raise RuntimeError(f"[X] Huấn luyện nhánh M5 thất bại với exit code: {ret_m5}!")
else:
    print(f"[✓] Huấn luyện nhánh M5 hoàn tất thành công trong {elapsed_m5:.2f}s ({elapsed_m5/60:.2f} phút)!")


# ==============================================================================
# 7. GIẢI PHÓNG DUNG LƯỢNG & XUẤT OUTPUT CHUẨN CHO KAGGLE SAVE VERSION
# ==============================================================================
import os
import shutil
from IPython.display import FileLink, display

print("=" * 80)
print("BƯỚC 7: GIẢI PHÓNG DUNG LƯỢNG VÀ XUẤT CHECKPOINTS VƯỢT TRỘI RA OUTPUT")
print("=" * 80)

# 1. Xóa tệp zip dở dang (nếu có) để thu hồi dung lượng đĩa trống
broken_zip = "/kaggle/working/task_6.3_checkpoints.zip"
if os.path.exists(broken_zip):
    os.remove(broken_zip)
    print("[✓] Đã xóa tệp zip dở dang để thu hồi dung lượng đĩa trống.")

# 2. Xóa các checkpoint phụ checkpoint_latest.pth để tiết kiệm ~7 GB
for branch in ["m3", "m4", "m5"]:
    b_dir = os.path.join(WORK_DIR, "DATASET", "checkpoints", branch)
    latest_file = os.path.join(b_dir, "checkpoint_latest.pth")
    if os.path.exists(latest_file):
        os.remove(latest_file)
        print(f"[✓] Đã dọn dẹp file phụ: {latest_file}")

# 3. Sao chép 3 model vượt trội và file nhật ký ra thẳng /kaggle/working/
output_files = []
for branch in ["m3", "m4", "m5"]:
    src_best = os.path.join(WORK_DIR, "DATASET", "checkpoints", branch, "crossdomain_model_best.pth")
    dst_best = f"/kaggle/working/{branch}_crossdomain_model_best.pth"
    if os.path.exists(src_best):
        shutil.copy2(src_best, dst_best)
        size_mb = os.path.getsize(dst_best) / (1024**2)
        print(f" [+] Đã sẵn sàng tại Output: {dst_best} ({size_mb:.2f} MB)")
        output_files.append(dst_best)

# Sao chép tệp nhật ký CSV
src_csv = os.path.join(WORK_DIR, "DATASET", "logs", "m6_training_history.csv")
dst_csv = "/kaggle/working/m6_training_history.csv"
if os.path.exists(src_csv):
    shutil.copy2(src_csv, dst_csv)
    print(f" [+] Đã sẵn sàng tại Output: {dst_csv}")
    output_files.append(dst_csv)

print("\n" + "=" * 80)
print("[✓] TOÀN BỘ KẾT QUẢ ĐÃ SẴN SÀNG AN TOÀN TRONG OUTPUT CỦA KAGGLE!")
print("👉 BÂY GIỜ BẠN CÓ THỂ BẤM: Save Version -> Quick Save -> Save Output for this version")
print("=" * 80)


In [ ]:
# ==============================================================================
# 7. ĐÓNG GÓI CHECKPOINT & HIỂN THỊ LINK TẢI VỀ MÁY TÍNH
# ==============================================================================
import os
import zipfile
from IPython.display import FileLink, display

print("=" * 80)
print("BƯỚC 7: ĐÓNG GÓI CHECKPOINTS VÀ NHẬT KÝ HUẤN LUYỆN MILESTONE 6")
print("=" * 80)

output_zip = "/kaggle/working/task_6.3_checkpoints.zip"
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # 1. Quét checkpoints của từng nhánh
    for branch in ["m3", "m4", "m5"]:
        b_dir = os.path.join(WORK_DIR, "DATASET", "checkpoints", branch)
        if not os.path.exists(b_dir):
            b_dir = os.path.join(WORK_DIR, "checkpoints", branch)
        if os.path.exists(b_dir):
            for f in os.listdir(b_dir):
                if f.endswith(".pth"):
                    full_p = os.path.join(b_dir, f)
                    arc_name = os.path.join("checkpoints", branch, f)
                    zf.write(full_p, arcname=arc_name)
                    print(f" [+] Đã thêm: {arc_name} ({os.path.getsize(full_p)/(1024**2):.2f} MB)")

    # 2. Quét nhật ký huấn luyện
    history_csv = os.path.join(WORK_DIR, "DATASET", "logs", "m6_training_history.csv")
    if os.path.exists(history_csv):
        zf.write(history_csv, arcname="m6_training_history.csv")
        print(f" [+] Đã thêm: m6_training_history.csv ({os.path.getsize(history_csv)} bytes)")

zip_size_mb = os.path.getsize(output_zip) / (1024 * 1024)
print(f"\n[✓] ĐÃ NÉN THÀNH CÔNG GÓI KẾT QUẢ: {output_zip} ({zip_size_mb:.2f} MB)")
print("👉 BẤM VÀO LIÊN KẾT BÊN DƯỚI ĐỂ TẢI VỀ MÁY TÍNH CÁ NHÂN:")
display(FileLink(output_zip))
